# autosearch smoke — one bounded live candidate (GPU T4 x2)

One arm (multi_agent), one repeat, config.one_experiment.yaml: one
paired calibration run + exactly one candidate experiment with a 10s
training budget. Validates the live path, not science. Expected wall
time ~20-35 min (incl. data prep). Requires the GEMINI_API_KEY secret
(checked BEFORE any expensive step).

In [ ]:
# 1. Clone project + deps (fail fast on git/pip problems).
import os, subprocess, sys
from pathlib import Path
WORK = Path('/kaggle/working')
PROJ = WORK / 'autosearch_kaggle'
os.chdir(WORK)  # never rmtree the directory the kernel is currently sitting in
if PROJ.exists():
    import shutil; shutil.rmtree(PROJ)
subprocess.run(['git', 'clone', '-q',
                'https://github.com/MokshonWork/autosearch_kaggle.git',
                str(PROJ)], check=True)
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print(subprocess.run(['git', 'log', '--oneline', '-2'],
                     capture_output=True, text=True).stdout)
print(subprocess.run(['nvidia-smi', '--list-gpus'],
                     capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rustbpe>=0.1.0', 'tiktoken>=0.11.0'], check=True)
# Informational only: the kernel's own torch is not used for training
# (subprocesses import torch themselves); never block on it.
try:
    import torch
    print('torch', torch.__version__, 'devices', torch.cuda.device_count())
except Exception as exc:
    print('kernel-process torch check failed (non-fatal):',
          type(exc).__name__, str(exc)[:200])
    import importlib.util
    print('torch spec:', importlib.util.find_spec('torch'))


In [ ]:
# 2. Secret — HARD requirement for the smoke (Gemini calls happen). Fail fast.
# NOTE: do NOT use kaggle_secrets.UserSecretsClient directly. Kaggle injects
# KAGGLE_URL_BASE=https://api.kaggle.com, but the UserSecretsService endpoint
# is served only by https://www.kaggle.com; against the injected host every
# request returns HTTP 400 with an empty body, which kaggle_web_client
# flattens into a generic ConnectionError (the "flaky backend" red herring).
# src/kaggle_secret.py talks to the working host directly and turns the
# server's answer into an actionable message:
#   * "not attached to this kernel"  -> attach it in the Kaggle UI
#   * 400 / empty body               -> the real routing/backend symptom
import os, sys
sys.path.insert(0, '/kaggle/working/autosearch_kaggle')
from src.kaggle_secret import (load_api_key, SecretNotAttachedError,
                               SecretAuthError, SecretBackendError)
try:
    load_api_key('GEMINI_API_KEY')
except SecretNotAttachedError as exc:
    print('SECRET NOT ATTACHED:', exc)
    raise
except (SecretAuthError, SecretBackendError) as exc:
    print('SECRET FETCH FAILED:', type(exc).__name__, exc)
    raise
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY secret missing or empty'
print('GEMINI_API_KEY loaded:', bool(os.environ.get('GEMINI_API_KEY')))


In [ ]:

# 2b. OPTIONAL model availability diagnostic (assumption A6).
# Disabled by default: a normal smoke run must not spend provider quota on
# probes. Set AUTOSEARCH_MODEL_PROBE=1 to enable this diagnostic manually.
# Probes EVERY candidate once. Only configured ids can fail; timeouts/request
# errors are NEVER treated as success, and extra candidates are informational.
# HTTP 429 means rate-limited/quota-exhausted, not proof the model is usable.
# Logic lives in src/kaggle_model_probe.py (unit-tested).
import os
if os.environ.get('AUTOSEARCH_MODEL_PROBE') != '1':
    print('Skipping optional model-availability probe (no network calls). '
          'Set AUTOSEARCH_MODEL_PROBE=1 to enable this diagnostic manually.')
else:
    import yaml
    from pathlib import Path
    from src.kaggle_model_probe import probe_models
    cfg = yaml.safe_load(Path('config.smoke.yaml').read_text())
    llm = cfg['llm']
    provider = llm.get('provider', 'gemini')
    api_env = llm.get('api_key_env', 'GEMINI_API_KEY')
    configured = sorted({v for v in llm.values()
                         if isinstance(v, str) and v.startswith('gemini-')})
    candidates = sorted({'gemini-3.8-flash', 'gemini-flash-latest',
                         'gemini-4-argon'} - set(configured))
    ids = configured + candidates
    print('probing model availability:', ids)
    probe_models(ids, configured, provider, os.environ[api_env])


In [ ]:
# 3. Nested repo/.git + data prep (skips if cache already present).
import subprocess, sys
from pathlib import Path
repo = Path('/kaggle/working/autosearch_kaggle/repo')
if not (repo / '.git').exists():
    for a in (['init', '-q'], ['add', '-A'],
              ['-c', 'user.email=k@kaggle', '-c', 'user.name=k',
               'commit', '-q', '-m', 't4-port baseline']):
        subprocess.run(['git', '-C', str(repo), *a], check=True)
    print('repo/.git re-initialized')
r = subprocess.run([sys.executable, 'prepare.py', '--num-shards', '4'],
                   cwd=str(repo), capture_output=True, text=True, timeout=3600)
print(r.stdout[-1200:])
print('prepare exit:', r.returncode)
assert r.returncode == 0, 'dataset/tokenizer prep failed'


In [ ]:
# 4. One bounded live candidate experiment. Full output saved to log files.
import subprocess, sys
from pathlib import Path
out = Path('/kaggle/working/smoke_results')
out.mkdir(parents=True, exist_ok=True)
p = subprocess.run(
    [sys.executable, 'evals/run_ab.py',
     '--config', 'config.one_experiment.yaml',
     '--repeats', '1', '--arms', 'multi_agent',
     '--out', str(out)],
    cwd='/kaggle/working/autosearch_kaggle',
    capture_output=True, text=True, timeout=4500)
(out / 'run_ab_stdout.log').write_text(p.stdout)
(out / 'run_ab_stderr.log').write_text(p.stderr)
print('run_ab exit:', p.returncode)
print('--- stdout tail ---')
print(p.stdout[-3000:])
print('--- stderr tail ---')
print(p.stderr[-2000:])
assert p.returncode == 0, 'run_ab failed — see logs above'


In [ ]:
# 5. Validate the one-candidate result: state, experiments, budgets, metrics.
import json, math, yaml
from pathlib import Path

out = Path('/kaggle/working/smoke_results')
mem = out / 'multi_agent' / 'run0' / 'memory'
cfg = yaml.safe_load(Path(
    '/kaggle/working/autosearch_kaggle/config.one_experiment.yaml').read_text())
api_cap = cfg['budget']['max_api_calls_total']
token_cap = cfg['budget']['max_tokens_total']

# 5a. Did the A/B runner (cell 4) produce its wiring evidence at all?
#     Cell 4 asserts run_ab exit 0, so if memory/ is missing everything below
#     is explained by the runner dying BEFORE writing any experiment.
required = {
    'state.json': mem / 'state.json',
    'experiments.jsonl': mem / 'experiments.jsonl',
    'lessons.md': mem / 'lessons.md',
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    print('smoke_results/memory directory listing:')
    if mem.exists():
        for p in sorted(mem.glob('*')):
            print('  ', p)
    print('MISSING: ' + ', '.join(missing))
    print('run_ab stdout -> /kaggle/working/smoke_results/run_ab_stdout.log')
    print('run_ab stderr -> /kaggle/working/smoke_results/run_ab_stderr.log')
    msg_lines = [
        'smoke run produced NO experiments: the A/B runner died',
        'before writing its memory (experiments.jsonl missing).',
        'Read /kaggle/working/smoke_results/run_ab_stderr.log',
        'AND the kernel output above. The first thing the runner',
        'does is calibration (Section 6.4): if the unmodified',
        'train.py fails there (status timeout/crash/nan) it aborts',
        'the whole run with e.g.',
        '    RuntimeError: calibration produced no valid runs',
        'and NO candidate experiment is ever written. Fix the training',
        'budget / data prep and re-push, re-run.',
    ]
    raise AssertionError(chr(10).join(msg_lines))
for p in sorted(mem.glob('*')):
    print('memory file:', p)

state = json.loads((mem / 'state.json').read_text())
exps = [json.loads(l) for l in
        (mem / 'experiments.jsonl').read_text().splitlines() if l.strip()]
print('state:', json.dumps({k: state.get(k) for k in (
    'round', 'experiments_done', 'best_val_bpb', 'baseline_bpb',
    'epsilon', 'stopped_reason')}, indent=2))
for e in exps:
    r = e['result']
    hid = e['hypothesis']['id']
    print(f"{hid} gpu{e['worker']} {r['status']} "
          f"val_bpb={r['val_bpb']} steps={r['steps']} wall={r['wall_seconds']}s")

checks = {
    'epsilon_set': state.get('epsilon') is not None,
    'baseline_finite': (state.get('baseline_bpb') is not None
                        and math.isfinite(state['baseline_bpb'])
                        and state['baseline_bpb'] > 0),
    'budget_stop': state.get('stopped_reason') in
                   ('max_rounds', 'max_experiments', 'no_improvement'),
    'one_candidate_limit': (state.get('experiments_done') == 1
                            and len(exps) == 1),
    'candidate_experiment_persisted': (
        len(exps) == 1
        and exps[0]['hypothesis']['id'].startswith('H-')
        and exps[0]['result']['hypothesis_id']
        == exps[0]['hypothesis']['id']),
    'api_calls_made': state.get('api_calls', 0) > 0,
    'api_budget_respected': state.get('api_calls', 0) <= api_cap,
    'token_budget_respected': state.get('api_tokens', 0) <= token_cap,
    'at_least_one_ok_run': any(
        e['result']['status'] == 'ok'
        and e['result']['val_bpb'] is not None
        and math.isfinite(e['result']['val_bpb'])
        and e['result']['val_bpb'] > 0
        and e['result']['steps'] > 0 for e in exps),
    'gpu_ids_valid': all(e['worker'] in (0, 1) for e in exps),
    'ab_report_exists': (out / 'ab_report.md').exists(),
}
summary = {
    'checks': checks,
    'state': {k: state.get(k) for k in (
        'round', 'experiments_done', 'baseline_bpb', 'best_val_bpb',
        'epsilon', 'calibration_values', 'stopped_reason', 'api_calls',
        'api_tokens')},
    'experiments': [f"{e['hypothesis']['id']}:{e['result']['status']}"
                    for e in exps],
}
(out / 'smoke_summary.json').write_text(json.dumps(summary, indent=2))
failed = [k for k, v in checks.items() if not v]
if failed:
    print('state summary:')
    print(json.dumps({k: state.get(k) for k in (
        'round', 'experiments_done', 'baseline_bpb', 'best_val_bpb',
        'epsilon', 'stopped_reason', 'api_calls', 'api_tokens')}, indent=2))
    print('experiments:')
    for e in exps:
        print(' ', e['hypothesis']['id'], e['result']['status'],
              e['result'].get('val_bpb'), e['result'].get('steps'),
              e['result'].get('wall_seconds'))
    persisted = checks['candidate_experiment_persisted']
    trained = checks['at_least_one_ok_run']
    print('candidate attempt persisted:', persisted)
    print('successful trained candidate:', trained)
    extra = ''
    if persisted and not trained:
        status = exps[0]['result']['status']
        extra = (f" (candidate attempt WAS persisted with status={status}, "
                 "but no successful trained candidate)")
        print('NOTE: the one-candidate attempt completed, but the candidate '
              'did not finish an ok training run; strict success checks fail.')
    raise AssertionError('one-candidate checks failed: '
                         + ', '.join(failed) + extra)
print(json.dumps(checks, indent=2))
print('ONE_CANDIDATE_COMPLETE_ALL_CHECKS_PASS')

